In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import sys
from sklearn.linear_model import LinearRegression

from helpers import pixel_to_cm_on_tablet
from helpers.metadata import RAIL_WIDTH_CM, AVOIDING_THRESHOLD, PARTICIPANT_INFO

In [2]:
CONDITIONS = ["ad", "ai"]
OUTPUT = "preprocessed"

### Join Individual Participants and Trials

In [ ]:
if not os.path.exists(OUTPUT):
    os.mkdir(OUTPUT)

for condition in CONDITIONS:
    df = pd.concat([pd.read_csv(file) for file in glob.iglob(f"data/*{condition}/*")])
    df.to_csv(os.path.join(OUTPUT, f"{condition}_raw.csv"), index=False)

### Preprocess Trials Get Necessary Data

In [23]:
def trial_wise_processing(df: pd.DataFrame):
    df["rtime"] = (df["timestamp"] - df["timestamp"].iloc[0]) / df["timestamp"].iloc[-1]
    df.index = pd.to_datetime(df["rtime"], unit="s")

    numeric_df = df.select_dtypes(include="number").resample("10ms").mean().interpolate()
    non_numeric_df = df.select_dtypes(exclude="number").resample("10ms").ffill()
    df = numeric_df.join(non_numeric_df)

    df.drop("rtime", axis=1, inplace=True)

    df["avoided"] = np.where(df["current_pos_x"] > (RAIL_WIDTH_CM * AVOIDING_THRESHOLD), 1, 0) 

    return df

In [ ]:
for condition in CONDITIONS:
    print(f"### processing {condition}")

    df = pd.read_csv(os.path.join(OUTPUT, f"{condition}_raw.csv"))

    ### Convert to pixels
    print("trajectory to cm")
    df["current_pos_x"] = pixel_to_cm_on_tablet(df["current_pos_x"], 0)
    df["current_pos_y"] = pixel_to_cm_on_tablet(df["current_pos_y"], 1)

    df["target_pos_x"] = pixel_to_cm_on_tablet(df["target_pos_x"], 0)
    df["target_pos_y"] = pixel_to_cm_on_tablet(df["target_pos_y"], 1)

    ### Drop columns
    print("drop button click columns")
    df.drop("left_button_pressed", axis=1, inplace=True)
    df.drop("middle_button_pressed", axis=1, inplace=True)
    df.drop("right_button_pressed", axis=1, inplace=True)

    ### Trial wise processing 
    print("processing trial wise")
    df = df.groupby(["participant_id", "trial_index"]).apply(trial_wise_processing, include_groups=False)

    ### drop duplicate timing columns
    print("drop duplicate timing columns")
    df.drop("timestamp", axis=1, inplace=True)

    ### round value 
    df["current_pos_x"] = df["current_pos_x"].round(3)
    df["current_pos_y"] = df["current_pos_y"].round(3)
    df["target_pos_x"] = df["target_pos_x"].round(3)
    df["target_pos_y"] = df["target_pos_y"].round(3)

    ### Save to csv
    print("saving to file")
    df.to_csv(os.path.join(OUTPUT, f"{condition}_processed.csv"), chunksize=10_000)

### processing ad
trajectory to cm
drop button click columns
processing trial wise


TypeError: DataFrame.groupby() got an unexpected keyword argument 'include_groups'

### Relevant 

In [53]:
def trial_wise_choose_relevant(df: pd.DataFrame):
    """Expecting single trial, so task and mapping should be the same"""
    row = None
    if df["task"].iloc[0] == "reaching":
        row = df.iloc[-1].copy()

    elif df["task"].iloc[0] == "avoiding":
        filtered = df.loc[df["avoided"] == 1]
        row = filtered.iloc[0].copy() if filtered.shape[0] > 0 else df.iloc[-1].copy()

    row.loc["error"] = (row.loc["current_pos_y"] - row.loc["target_pos_y"]).round(3)
    row.loc["error_abs"] = (row.loc["current_pos_y"] - row.loc["target_pos_y"]).round(3)

    return row

In [54]:
for condition in CONDITIONS:
    print(f"### processing {condition}")

    df = pd.read_csv(os.path.join(OUTPUT, f"{condition}_processed.csv"))
    
    ### Trial wise processing 
    print("processing trial wise")
    df = df.groupby(["participant_id", "trial_index"]).apply(trial_wise_choose_relevant, include_groups=False)

    ### Save to csv
    print("saving to file")
    df.to_csv(os.path.join(OUTPUT, f"{condition}_relevant.csv"), chunksize=1_000)

### processing ad
processing trial wise
saving to file
### processing ai
processing trial wise
saving to file


### Compute Block Metrics

In [55]:
def get_block_metric(df: pd.DataFrame):
    return pd.Series({
        "error_mean": df["error"].mean(),
        "error_std": df["error"].std(),
        "error_abs_mean": df["error_abs"].mean(),
        "error_abs_std": df["error_abs"].std(),
        "cursor_y_mean":  df["current_pos_y"].mean(), 
        "cursor_y_std": df["current_pos_y"].std(),
        "task_mapping": df["mapping"].iloc[-1]
    })

In [56]:
for condition in CONDITIONS:
    print(f"### processing {condition}")

    df = pd.read_csv(os.path.join(OUTPUT, f"{condition}_relevant.csv"))
    
    ### Trial wise processing 
    print("processing trial wise")
    df =  df.groupby(["participant_id", "phase", "task", "block", "target_pos_y"]).apply(get_block_metric,  include_groups=False)

    ### Rounding the values
    df = df.round({
        "error_mean": 3, 
        "error_std": 3,
        "error_abs_mean": 3, 
        "error_abs_std": 3,
        "cursor_y_mean": 3, 
        "cursor_y_std": 3
    })
    
    ### Save to csv
    print("saving to file")
    df.to_csv(os.path.join(OUTPUT, f"{condition}_metrics.csv"), chunksize=1_000)

### processing ad
processing trial wise
saving to file
### processing ai
processing trial wise
saving to file


### Outlier detection

In [ ]:
"""total_outliers = 0
for condition in CONDITIONS:
    print(f"### processing {condition}")


    target_metric = pd.read_csv(os.path.join(OUTPUT, f"{condition}_metrics.csv"))
    relevant = pd.read_csv(os.path.join(OUTPUT, f"{condition}_relevant.csv"))

    ### Trial wise processing 
    print("processing trial wise")
    for target_data in relevant.groupby(["participant_id", "target_pos_y", "phase", "block", "task"]): # each entry is one trial here
        keys = target_data[0]
        
        part_metric = target_metric[(target_metric["participant_id"] == keys[0]) & (target_metric["target_pos_y"] == keys[1]) & (target_metric["phase"] == keys[2]) & (target_metric["block"] == keys[3]) & (target_metric["task"] == keys[4])]
        mean, std = part_metric["error_mean"].iloc[0], part_metric["error_std"].iloc[0]     
        
        outliers = target_data[1][(target_data[1]["current_pos_y"] < mean - 3 * std) | (target_data[1]["current_pos_y"] > mean + 3 * std)]
        total_outliers += len(outliers)

print(total_outliers)
    ### Save to csv
#    print("saving to file")
#    df.to_csv(os.path.join(OUTPUT, f"{condition}_participant_metrics.csv"), chunksize=1_000)"""

### processing ad
processing trial wise
### processing ai
processing trial wise
6606


### Get Participant Metadata

In [ ]:
def get_participant_metrics(df: pd.DataFrame):
    """Expects single participant data"""

    y = df.loc[:, "cursor_y_mean"].values
    X = df.loc[:, "target_pos_y"].values

    regressor = LinearRegression()
    regressor.fit(X[:, np.newaxis], y)
    

    theta = 0.25
    threshold_upper = theta
    threshold_lower = -theta

    slope = regressor.coef_[0]
    corr = np.corrcoef(X, y)[0, 1]

    task_mapping = df.loc[:, "task_mapping"].iloc[0]

    reversed_mapping = lambda task_mapping: "reversed" if task_mapping == "direct" else "direct"

    mapping = "no mapping" # random, no mapping, direct, reversed
    if corr > threshold_lower and corr < threshold_upper:    
        mapping = "random"

    else:
        if slope < threshold_lower:
            mapping = reversed_mapping(task_mapping)
        elif slope > threshold_upper:
            mapping = task_mapping

    id = df["participant_id"].iloc[0]
    info = PARTICIPANT_INFO[id]

    return pd.Series({
        "age": info["age"],
        "handedness": info["handedness"],
        "gender": info["gender"],
        "group": info["group"],
        "corr": corr,
        "reg_slope": regressor.coef_[0],
        "reg_intercept": regressor.intercept_,
        "natural_mapping": mapping
    })
    return series

In [88]:
for condition in CONDITIONS:
    print(f"### processing {condition}")

    df = pd.read_csv(os.path.join(OUTPUT, f"{condition}_metrics.csv"))
    
    ### Trial wise processing 
    print("processing trial wise")
    df =  df.groupby(["participant_id"]).apply(get_participant_metrics)
    
    ### Save to csv
    print("saving to file")
    df.to_csv(os.path.join(OUTPUT, f"{condition}_participant_metrics.csv"), chunksize=1_000)

### processing ad
processing trial wise
saving to file
### processing ai
processing trial wise
saving to file


/tmp/ipykernel_13424/3319873262.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df =  df.groupby(["participant_id"]).apply(get_participant_metrics)
/tmp/ipykernel_13424/3319873262.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df =  df.groupby(["participant_id"]).apply(get_participant_metrics)
